In [6]:
import random, unicodedata, torch, faiss, json
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

torch.set_num_threads(2)

print("Chargement des modèles...")
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cpu")
model_name = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(
    model_name, torch_dtype=torch.float32
)
qa_model.eval()

with open("ensa_kb.json", "r", encoding="utf-8") as f:
    ensa_data = json.load(f)

index = faiss.read_index("ensa_faiss_v3.index")
print(f"Prêt — {len(ensa_data)} entrées · {index.ntotal} vecteurs")

Chargement des modèles...


C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Prêt — 320 entrées · 320 vecteurs


In [7]:
SEUIL_COSINUS = 0.40

# ── Normalisation ─────────────────────────────────────────────
def normaliser(texte):
    return unicodedata.normalize('NFD', texte)\
                      .encode('ascii', 'ignore')\
                      .decode('utf-8')

# ── Messages sociaux ──────────────────────────────────────────
REPONSES_SOCIALES = {
    "salutation": [
        "Hello! I'm the ENSA Fès intelligent assistant. Feel free to ask me anything about programs, admissions, clubs, or contacts!",
        "Hi there! Welcome to the ENSA Fès chatbot. What would you like to know?",
        "Bonjour! Je suis l'assistant intelligent de l'ENSA Fès. Comment puis-je vous aider?",
        "Salut! Bienvenue sur le chatbot de l'ENSA Fès. Posez-moi vos questions!",
    ],
    "comment_ca_va": [
        "I'm doing great, thank you! Ready to help you with anything about ENSA Fès.",
        "All systems running smoothly! What can I help you with today?",
        "Très bien merci! Je suis opérationnel et prêt à vous aider.",
    ],
    "merci": [
        "You're very welcome! Don't hesitate to ask more questions.",
        "Happy to help! Is there anything else you'd like to know?",
        "De rien! C'est avec plaisir. Avez-vous d'autres questions?",
    ],
    "au_revoir": [
        "Goodbye! Good luck with your studies at ENSA Fès!",
        "Au revoir! Bonne continuation et bonne chance!",
        "See you soon! Don't hesitate to come back anytime.",
    ],
    "identite": [
        "I'm an intelligent chatbot built for ENSA Fès using a RAG pipeline — combining a semantic BERT retriever and a RoBERTa reader.",
        "I'm the ENSA Fès AI Assistant! I can answer questions about programs, admissions, clubs, internships, contacts, and much more!",
        "Je suis l'assistant intelligent de l'ENSA Fès, basé sur une architecture RAG.",
    ],
    "blague": [
        "Why do programmers prefer dark mode? Because light attracts bugs! Anyway, how can I help you with ENSA Fès?",
        "I'd tell you a joke about engineers, but I'm still debugging it. What can I help you with?",
    ],
    "felicitations": [
        "Thank you! I'm just doing my job. What else can I help you with?",
        "That's very kind! I'm here to make your ENSA Fès experience easier.",
        "Merci beaucoup! Avez-vous d'autres questions?",
    ],
    "aide": [
        "Sure! I can help you with:\n- Engineering programs & specializations\n- Admission process & CNC exam\n- Internships & PFE\n- Student clubs & activities\n- Campus contacts & location\n- Tuition fees & scholarships\nWhat do you need?",
        "Bien sûr! Je peux vous aider avec les filières, admissions, stages, clubs, contacts et bien plus!",
    ],
}

# Mots qui indiquent une vraie question technique — jamais sociale
MOTS_TECHNIQUES = {
    "machine", "learning", "deep", "neural", "network", "algorithm",
    "program", "programming", "software", "engineering", "computer",
    "data", "science", "artificial", "intelligence", "python", "code",
    "internship", "admission", "club", "fees", "tuition", "exam",
    "study", "degree", "master", "bachelor", "professor", "course",
    "module", "semester", "grade", "scholarship", "campus", "library",
    "research", "project", "pfe", "cnc", "ensa", "fes", "university",
    "school", "department", "faculty", "student", "apply", "register",
    "restaurant", "weather", "football", "sport", "news", "world",
    "what","where","when","how","why","who","which","explain","tell",
    "describe","list","give","show","define","compare","difference"
}

def detecter_social(query):
    q = query.lower().strip().rstrip("!?.,")
    mots = q.split()

    # Si la question contient un mot technique → jamais sociale
    if any(mot in MOTS_TECHNIQUES for mot in mots):
        return None

    # Salutations — message court
    if len(mots) <= 5 and any(s in q for s in [
        "hello","hi","hey","bonjour","salut","salam",
        "bonsoir","good morning","good evening","ahlan"
    ]):
        return random.choice(REPONSES_SOCIALES["salutation"])

    # Comment ça va — message court
    if len(mots) <= 6 and any(s in q for s in [
        "how are you","comment ca va","comment vas","ca va",
        "what's up","labas","kif"
    ]):
        return random.choice(REPONSES_SOCIALES["comment_ca_va"])

    # Merci — message court
    if len(mots) <= 5 and any(s in q for s in [
        "thank","merci","choukran","appreciate",
        "well done","good job"
    ]):
        return random.choice(REPONSES_SOCIALES["merci"])

    # Félicitations — message très court
    if len(mots) <= 4 and any(s in q for s in [
        "bravo","excellent","perfect","parfait",
        "super","awesome","impressive"
    ]):
        return random.choice(REPONSES_SOCIALES["felicitations"])

    # Au revoir — message court
    if len(mots) <= 5 and any(s in q for s in [
        "bye","goodbye","au revoir","bonne journee",
        "see you","bslama","ciao"
    ]):
        return random.choice(REPONSES_SOCIALES["au_revoir"])

    # Identité — phrases spécifiques
    if any(s in q for s in [
        "who are you","what are you","tu es qui",
        "introduce yourself","présente toi"
    ]):
        return random.choice(REPONSES_SOCIALES["identite"])

    # Ce que tu peux faire — phrases spécifiques
    if any(s in q for s in [
        "what can you do","que peux-tu faire",
        "what do you know","how can you help me"
    ]):
        return random.choice(REPONSES_SOCIALES["aide"])

    # Blague — phrases spécifiques
    if any(s in q for s in [
        "tell me a joke","make me laugh",
        "fais moi rire","raconte une blague"
    ]):
        return random.choice(REPONSES_SOCIALES["blague"])

    # Aide seul mot
    if q in ["help","aide","assist"]:
        return random.choice(REPONSES_SOCIALES["aide"])

    return None
    
# ── Retriever ─────────────────────────────────────────────────
def retrieve(query, top_k=3):
    qvec = embedding_model.encode([normaliser(query)]).astype(np.float32)
    faiss.normalize_L2(qvec)
    scores, indices = index.search(qvec, top_k)
    return [{
        "rank": i + 1,
        "score_cosinus": float(scores[0][i]),
        "answer": ensa_data[idx]["answer"],
        "question": ensa_data[idx]["question"],
        "category": ensa_data[idx]["category"]
    } for i, idx in enumerate(indices[0]) if idx < len(ensa_data)]

# ── Reader RoBERTa ────────────────────────────────────────────
def read_answer(question, context):
    inputs = tokenizer(
        question, context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1
    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end],
        skip_special_tokens=True
    )
    score = float(torch.softmax(outputs.start_logits, dim=1).max())
    return answer, score

# ── AMÉLIORATION 1 : Réponse complète ────────────────────────
def construire_reponse_complete(span_extrait, passage_complet, score_qa):
    """
    Combine le span extrait par RoBERTa avec le passage complet
    pour donner une réponse plus riche et informative.
    
    Logique :
    - Si score_qa élevé (>0.7) : span précis + passage complet
    - Si score_qa moyen (0.4-0.7) : passage complet directement
    - Si score_qa faible (<0.4) : passage complet avec avertissement
    """
    if score_qa >= 0.7:
        # RoBERTa très confiant → span + contexte complet
        if span_extrait and span_extrait not in passage_complet[:len(span_extrait)+5]:
            return f"{span_extrait}\n\n{passage_complet}"
        return f"{passage_complet}"
    elif score_qa >= 0.4:
        # Confiance modérée → passage complet directement
        return passage_complet
    else:
        # Confiance faible → passage avec note
        return f"{passage_complet}\n\n*Note: This is the most relevant information I found. For more details, contact contact@ensa-fes.ma*"

# ── Score confiance ───────────────────────────────────────────
def calculer_confiance(score_cosinus, score_qa):
    if score_cosinus < SEUIL_COSINUS:
        return 0
    return min(round((0.6 * score_cosinus + 0.4 * score_qa) * 100), 99)

# ── Pipeline principal ────────────────────────────────────────
def chatbot_ensa(query):
    # ÉTAPE 0 : Messages sociaux
    reponse_sociale = detecter_social(query)
    if reponse_sociale:
        return reponse_sociale, 99, "social", "Built-in social response.", []

    # ÉTAPE 1 : Retrieval
    passages = retrieve(query, top_k=3)
    if not passages:
        return "Internal error.", 0, "error", "", []

    meilleur = passages[0]

    # ÉTAPE 2 : Vérification seuil
    if meilleur["score_cosinus"] < SEUIL_COSINUS:
        return (
            "I couldn't find relevant information in my ENSA Fès database. "
            "Please contact contact@ensa-fes.ma or visit https://ensaf.ac.ma",
            0, "hors_kb", "No relevant passage found.", passages
        )

    # ÉTAPE 3 : Extraction RoBERTa
    span, score_qa = read_answer(query, meilleur["answer"])

    # ÉTAPE 4 : Réponse complète (AMÉLIORATION 1)
    reponse = construire_reponse_complete(span, meilleur["answer"], score_qa)

    # ÉTAPE 5 : Score confiance
    confiance = calculer_confiance(meilleur["score_cosinus"], score_qa)

    return (
        reponse,
        confiance,
        meilleur["category"],
        meilleur["answer"],
        passages  # top-3 pour affichage (AMÉLIORATION 3)
    )

print("Pipeline amélioré v1 prêt !")

Pipeline amélioré v1 prêt !


In [8]:
questions_test = [
    "What engineering programs are available at ENSA Fes?",
    "Are internships mandatory at ENSA Fes?",
    "What clubs are available at ENSA Fes?",
    "Hello",
    "Who are you?",
    "Thank you!",
]

print("TEST RÉPONSES COMPLÈTES\n" + "="*60)
for q in questions_test:
    res = chatbot_ensa(q)
    reponse, confiance, categorie = res[0], res[1], res[2]
    print(f"\nQ: {q}")
    print(f"Catégorie : {categorie} | Confiance : {confiance}%")
    print(f"Réponse :\n{reponse}")
    print("-"*60)

TEST RÉPONSES COMPLÈTES

Q: What engineering programs are available at ENSA Fes?
Catégorie : filieres | Confiance : 79%
Réponse :
ENSA Fès offers 11 initial training engineering specializations: Computer Engineering (GI), Networked Systems and Cybersecurity (GSCSI), Digital Development and Cybersecurity (GDNC), Energy Engineering and Intelligent Systems (GESI), Industrial Engineering, Mechanical Engineering (GM), Mechatronics Engineering, Embedded Systems and AI (ISEAI), Data Science and AI Engineering (ISDAI), IT Engineering with AI and Digital Trust (IIIACN), and Software Engineering and AI (ILIA). All are preceded by a 2-year Integrated Preparatory Cycle.
------------------------------------------------------------

Q: Are internships mandatory at ENSA Fes?
Catégorie : stages | Confiance : 77%
Réponse :
Yes. Internships are mandatory at ENSA Fès and are an integral part of the curriculum. There are three types: a worker internship (stage ouvrier) in Year 2 of the CPI, a technical in

In [2]:
from ddgs import DDGS

def fallback_web(query):
    """
    Déclenché quand score_cosinus < SEUIL (question hors KB).
    Cherche sur le web une réponse approximative.
    Confiance toujours 30% — signale que la source est externe.
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(
                query,
                max_results=3
            ))
        if results:
            meilleur = results[0]
            reponse = meilleur["body"][:400]
            source = meilleur["title"]
            return {
                "reponse": f"{reponse}\n\n*Web source : {source}*",
                "confiance": 30,
                "category": "web_fallback",
                "passage": f"Web : {meilleur['href']}"
            }
    except Exception as e:
        print(f"Fallback échoué : {e}")
    return None


# Seuil relevé à 0.45 pour éviter les faux positifs
SEUIL_COSINUS = 0.45

def chatbot_ensa_v2(query):
    # ÉTAPE 0 : Messages sociaux
    reponse_sociale = detecter_social(query)
    if reponse_sociale:
        return reponse_sociale, 99, "social", "Built-in social response.", []

    # ÉTAPE 1 : Retrieval
    passages = retrieve(query, top_k=3)
    if not passages:
        return "Internal error.", 0, "error", "", []

    meilleur = passages[0]

    # ÉTAPE 2 : Vérification seuil → fallback si hors KB
    if meilleur["score_cosinus"] < SEUIL_COSINUS:
        fb = fallback_web(query)
        if fb:
            return (
                fb["reponse"],
                fb["confiance"],
                fb["category"],
                fb["passage"],
                passages
            )
        return (
            "I couldn't find relevant information. "
            "Please contact contact@ensa-fes.ma or visit https://ensaf.ac.ma",
            0, "hors_kb", "No relevant passage found.", passages
        )

    # ÉTAPE 3 : Extraction RoBERTa
    span, score_qa = read_answer(query, meilleur["answer"])

    # ÉTAPE 4 : Réponse complète
    reponse = construire_reponse_complete(span, meilleur["answer"], score_qa)

    # ÉTAPE 5 : Score confiance
    confiance = calculer_confiance(meilleur["score_cosinus"], score_qa)

    return (
        reponse,
        confiance,
        meilleur["category"],
        meilleur["answer"],
        passages
    )

print("Pipeline v2 avec fallback prêt !")

Pipeline v2 avec fallback prêt !


In [9]:
questions_hors_kb = [
    "What is the weather in Fes today?",
    "Who won the World Cup 2022?",
    "What is machine learning?",
    "What restaurants are near ENSA Fes?",
]

print("TEST FALLBACK WEB\n" + "="*60)
for q in questions_hors_kb:
    res = chatbot_ensa_v2(q)
    reponse, confiance, categorie = res[0], res[1], res[2]
    print(f"\nQ: {q}")
    print(f"Catégorie : {categorie} | Confiance : {confiance}%")
    print(f"Réponse : {reponse[:200]}...")
    print("-"*60)

TEST FALLBACK WEB

Q: What is the weather in Fes today?
Catégorie : scolarite | Confiance : 62%
Réponse : The academic year at ENSA Fès is divided into two semesters. Semester 1 runs from September to January (exams in January), and Semester 2 runs from February to June (exams in June). Catch-up sessions ...
------------------------------------------------------------

Q: Who won the World Cup 2022?
Catégorie : web_fallback | Confiance : 30%
Réponse : The 2022 FIFA World Cup final was the final match of the 2022 FIFA World Cup , the 22nd edition of FIFA 's competition for men's national football ...

*Web source : 2022 FIFA World Cup final - Wikipe...
------------------------------------------------------------

Q: What is machine learning?
Catégorie : web_fallback | Confiance : 30%
Réponse : Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and 

In [10]:
questions_test = [
    # Dans KB — doivent répondre normalement
    "How to get admitted to ENSA Fes?",
    "What clubs are available at ENSA Fes?",
    "Are internships mandatory at ENSA Fes?",
    # Hors KB — doivent déclencher fallback web
    "What is the weather in Fes today?",
    "Who won the World Cup 2022?",
    "What is machine learning?",
    "What restaurants are near ENSA Fes?",
    # Social
    "Hello",
    "Thank you!",
]

print("TEST COMPLET v2\n" + "="*65)
for q in questions_test:
    res = chatbot_ensa_v2(q)
    reponse, confiance, categorie = res[0], res[1], res[2]
    badge = "VERT  " if confiance >= 70 else "ORANGE" if confiance >= 40 else "ROUGE "
    print(f"\n[{badge}] {confiance}% | {categorie}")
    print(f"Q: {q}")
    print(f"R: {reponse[:150]}...")
    print("-"*65)

TEST COMPLET v2

[VERT  ] 71% | admissions
Q: How to get admitted to ENSA Fes?
R: Admission to ENSA Fès is done through the National Common Entrance Exam (CNC) for scientific baccalaureate holders or students who have completed prep...
-----------------------------------------------------------------

[VERT  ] 83% | clubs
Q: What clubs are available at ENSA Fes?
R:  Computer Science Club, Robotics Club, Electronics Club, Sports Club, Cultural Club, Sustainable Development Club, Entrepreneurship Club

ENSA Fès has...
-----------------------------------------------------------------

[VERT  ] 77% | stages
Q: Are internships mandatory at ENSA Fes?
R: Yes. Internships are mandatory at ENSA Fès and are an integral part of the curriculum. There are three types: a worker internship (stage ouvrier) in Y...
-----------------------------------------------------------------

[ORANGE] 62% | scolarite
Q: What is the weather in Fes today?
R: The academic year at ENSA Fès is divided into two semester

In [16]:
def formater_sources(passages, categorie, confiance):
    """
    Génère un HTML propre affichant les top-3 passages récupérés
    avec leurs scores — montre la transparence du RAG pipeline.
    """
    if not passages or categorie in ["social", "hors_kb", "web_fallback", "error"]:
        return ""

    couleurs_cat = {
        "admissions":   ("#e8f0fe", "#1a73e8"),
        "filieres":     ("#e6f4ea", "#1e8e3e"),
        "stages":       ("#fce8b2", "#f29900"),
        "clubs":        ("#fce8f0", "#d93025"),
        "contacts":     ("#f3e8fd", "#9334e6"),
        "scolarite":    ("#e8f5e9", "#2e7d32"),
        "departements": ("#fff3e0", "#e65100"),
        "evenements":   ("#e0f7fa", "#006064"),
        "services":     ("#fafafa", "#424242"),
    }

    html = """
    <div style="margin-top:12px;">
        <div style="font-size:12px;font-weight:600;color:#003580;
            margin-bottom:8px;letter-spacing:0.5px;">
            TOP-3 RETRIEVED SOURCES
        </div>
    """

    for i, p in enumerate(passages[:3]):
        score = p["score_cosinus"]
        cat = p.get("category", "unknown")
        bg, border = couleurs_cat.get(cat, ("#f5f5f5", "#9e9e9e"))

        # Barre de score visuelle
        barre_width = int(score * 100)
        barre_color = "#1D9E75" if score >= 0.6 else \
                      "#e67e22" if score >= 0.45 else "#c0392b"

        # Rang medal
        medal = ["1st", "2nd", "3rd"][i]

        html += f"""
        <div style="background:{bg};border-left:3px solid {border};
            border-radius:8px;padding:10px 12px;margin-bottom:8px;">
            <div style="display:flex;justify-content:space-between;
                align-items:center;margin-bottom:6px;">
                <span style="font-size:11px;font-weight:700;
                    color:{border};">{medal} · {cat.upper()}</span>
                <span style="font-size:11px;font-weight:600;
                    color:{barre_color};">cos = {score:.4f}</span>
            </div>
            <div style="font-size:12px;color:#333;margin-bottom:6px;
                line-height:1.4;">
                {p['question'][:90]}{'...' if len(p['question']) > 90 else ''}
            </div>
            <div style="background:#e0e0e0;border-radius:4px;
                height:4px;width:100%;">
                <div style="background:{barre_color};height:4px;
                    border-radius:4px;width:{barre_width}%;
                    transition:width 0.3s;">
                </div>
            </div>
        </div>
        """

    html += "</div>"
    return html


# Test
print("TEST AFFICHAGE SOURCES\n" + "="*50)
res = chatbot_ensa_v2("What engineering programs are available at ENSA Fes?")
reponse, confiance, categorie, passage, passages = res
html_sources = formater_sources(passages, categorie, confiance)
print(f"Réponse : {reponse[:100]}...")
print(f"Sources HTML généré : {len(html_sources)} caractères")
print("OK — prêt pour l'interface !")

TEST AFFICHAGE SOURCES
Réponse : ENSA Fès offers 11 initial training engineering specializations: Computer Engineering (GI), Networke...
Sources HTML généré : 3316 caractères
OK — prêt pour l'interface !


In [11]:
import gradio as gr
import os

_dir = os.getcwd()

def repondre_v2(question, historique):
    if not question.strip():
        return historique, "", "", "", ""

    res = chatbot_ensa_v2(question)
    reponse, confiance, categorie, passage, passages = res

    if confiance >= 70:
        couleur = "#1D9E75"
        label = f"✓ High confidence — {confiance}%"
    elif confiance >= 40:
        couleur = "#e67e22"
        label = f"~ Moderate confidence — {confiance}%"
    else:
        couleur = "#c0392b"
        label = f"! Low confidence — {confiance}%"

    badge_html = f"""
    <div style="margin-top:8px;">
        <div style="display:inline-flex;align-items:center;gap:8px;
            padding:8px 18px;background:{couleur};color:white;
            border-radius:24px;font-size:13px;font-weight:600;">
            {label}
        </div>
        <div style="margin-top:5px;font-size:12px;color:#5a7fa8;padding-left:4px;">
            Category : <strong>{categorie}</strong>
        </div>
    </div>
    """

    sources_html = formater_sources(passages, categorie, confiance)
    historique = historique or []
    historique.append({"role": "user", "content": question})
    historique.append({"role": "assistant", "content": reponse})
    return historique, badge_html, passage, sources_html, ""


css = """
/* ─── Base ─────────────────────────────────────── */
.gradio-container {
    max-width: 900px !important;
    margin: auto !important;
    background: #eef2f8 !important;
    padding: 18px !important;
}
footer { display: none !important; }

/* ─── Supprimer TOUS les wrappers Gradio parasites ─ */
.gradio-container > .contain,
.gradio-container > .contain > div,
.gradio-container > div > div {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    padding: 0 !important;
    gap: 0 !important;
}

/* ─── Header ────────────────────────────────────── */
#ensa-header {
    background: linear-gradient(135deg, #001845 0%, #003075 55%, #0050aa 100%);
    border-radius: 20px;
    padding: 22px 28px;
    margin-bottom: 14px;
    display: flex;
    align-items: center;
    gap: 20px;
    box-shadow: 0 4px 20px rgba(0,24,80,0.22);
}
#ensa-header .logo-wrap {
    width: 72px; height: 72px;
    background: white; border-radius: 14px;
    display: flex; align-items: center; justify-content: center;
    flex-shrink: 0; overflow: hidden; padding: 5px;
}
#ensa-header .logo-wrap img { width:100%; height:100%; object-fit:contain; }
#ensa-header .logo-fallback { font-size:18px; font-weight:900; color:#003580; }
#ensa-header .htext { flex: 1; }
#ensa-header h1 { font-size:20px; font-weight:700; margin:0 0 4px; color:#fff; }
#ensa-header .sub { font-size:11.5px; color:rgba(255,255,255,0.7); margin:0 0 9px; }
#ensa-header .pills { display:flex; flex-wrap:wrap; gap:6px; }
#ensa-header .pill {
    background:rgba(255,255,255,0.13); border:1px solid rgba(255,255,255,0.28);
    color:rgba(255,255,255,0.92); font-size:9.5px; font-weight:600;
    padding:3px 10px; border-radius:20px;
}
#ensa-header .stat { text-align:right; flex-shrink:0; }
#ensa-header .stat .n { font-size:32px; font-weight:800; color:#fff; line-height:1; }
#ensa-header .stat .l { font-size:9px; color:rgba(255,255,255,0.85); letter-spacing:.7px; text-transform:uppercase; margin-top:3px; }
#ensa-header .stat .s { font-size:10px; color:rgba(255,255,255,0.55); margin-top:3px; }

/* ─── Quick Questions box ────────────────────────── */
#qq-wrap {
    background: white;
    border-radius: 16px;
    border: 1.5px solid #dce8f5;
    padding: 16px 18px 14px;
    margin-bottom: 14px;
    box-shadow: 0 2px 10px rgba(0,53,128,0.06);
}
#qq-wrap .qq-title {
    font-size: 10px; font-weight: 700; color: #003580;
    letter-spacing: 1.2px; text-transform: uppercase; margin-bottom: 12px;
}

/* Les boutons Gradio DANS la grille */
#qq-wrap .gr-button-group,
#qq-wrap > div { background: transparent !important; border: none !important; padding: 0 !important; }

/* Grille 2 colonnes sur les rows Gradio */
#qq-row1, #qq-row2, #qq-row3 {
    display: grid !important;
    grid-template-columns: 1fr 1fr !important;
    gap: 8px !important;
    margin-bottom: 8px !important;
}
#qq-row1 > *, #qq-row2 > *, #qq-row3 > * {
    flex: unset !important;
    width: 100% !important;
    min-width: 0 !important;
}

/* Style des boutons Gradio quick questions */
#btn-s1, #btn-s2, #btn-s3, #btn-s4, #btn-s5, #btn-s6 {
    background: #f0f4f9 !important;
    border: 1.5px solid #d0e1f5 !important;
    border-radius: 10px !important;
    color: #003580 !important;
    font-size: 12.5px !important;
    font-weight: 500 !important;
    padding: 11px 16px !important;
    width: 100% !important;
    text-align: center !important;
    transition: all 0.15s !important;
    box-shadow: none !important;
}
#btn-s1:hover, #btn-s2:hover, #btn-s3:hover,
#btn-s4:hover, #btn-s5:hover, #btn-s6:hover {
    background: #003580 !important;
    color: white !important;
    border-color: #003580 !important;
}

/* Reset button */
#btn-reset {
    background: transparent !important;
    border: 1px solid #c0d4ea !important;
    border-radius: 8px !important;
    color: #8aabcc !important;
    font-size: 11px !important;
    padding: 5px 16px !important;
    box-shadow: none !important;
    float: right;
    margin-top: 4px !important;
}
#btn-reset:hover {
    background: #fee2e2 !important;
    border-color: #f87171 !important;
    color: #dc2626 !important;
}

/* ─── Chat panel ─────────────────────────────────── */
#chat-wrap {
    background: white;
    border-radius: 16px;
    border: 1.5px solid #dce8f5;
    padding: 16px;
    box-shadow: 0 2px 12px rgba(0,53,128,0.06);
    margin-bottom: 14px;
}
/* Supprimer fond/border du chatbot Gradio lui-même */
#chatwindow {
    border-radius: 10px !important;
    border: 1px solid #eef2f8 !important;
    background: #f8fafd !important;
}
#chatwindow > div { background: transparent !important; }

#send-btn {
    background: #003580 !important; color: white !important;
    border-radius: 10px !important; font-weight: 700 !important;
    font-size: 14px !important; border: none !important;
    min-width: 90px !important; box-shadow: none !important;
}
#send-btn:hover { background: #0057b8 !important; }

/* ─── How It Works ───────────────────────────────── */
#hiw-wrap {
    background: linear-gradient(135deg, #001845 0%, #003075 100%);
    border-radius: 16px; padding: 20px 24px; margin-bottom: 14px;
}
#hiw-wrap .hiw-title {
    font-size: 10px; font-weight: 700; color: rgba(255,255,255,0.95);
    letter-spacing: 1.2px; text-transform: uppercase;
    margin-bottom: 14px; padding-bottom: 10px;
    border-bottom: 1px solid rgba(255,255,255,0.18);
}
#hiw-wrap .hiw-grid { display:grid; grid-template-columns:1fr 1fr; gap:16px; }
#hiw-wrap .hiw-item { display:flex; align-items:flex-start; gap:12px; }
#hiw-wrap .hiw-num {
    width:30px; height:30px; border-radius:50%;
    background:rgba(255,255,255,0.2); border:1.5px solid rgba(255,255,255,0.4);
    color:#fff; font-size:13px; font-weight:700;
    display:flex; align-items:center; justify-content:center; flex-shrink:0;
}
#hiw-wrap .hiw-text strong { display:block; font-size:12.5px; font-weight:600; color:#fff; margin-bottom:3px; }
#hiw-wrap .hiw-text span { font-size:11px; color:rgba(255,255,255,0.72); line-height:1.45; }

/* ─── Footer ─────────────────────────────────────── */
#ensa-footer {
    text-align:center; font-size:11px; color:#8da9c4;
    padding-top:12px; border-top:1px solid #dce8f5;
}
"""

with gr.Blocks(css=css) as demo:

    # ── HEADER ──────────────────────────────────────────────
    gr.HTML(f"""
    <div id="ensa-header">
        <div class="logo-wrap">
            <img src="/file={_dir}/logo_ensa.PNG"
                 onerror="this.style.display='none';this.nextElementSibling.style.display='block';"
                 alt="ENSA Logo">
            <span class="logo-fallback" style="display:none;">ENSA</span>
        </div>
        <div class="htext">
            <h1>ENSA Fès — Intelligent Chatbot</h1>
            <p class="sub">École Nationale des Sciences Appliquées de Fès &nbsp;·&nbsp; Université Sidi Mohamed Ben Abdellah</p>
            <div class="pills">
                <span class="pill">RAG PIPELINE</span>
                <span class="pill">BERT RETRIEVER</span>
                <span class="pill">RoBERTa READER</span>
                <span class="pill">320 QA PAIRS</span>
            </div>
        </div>
        <div class="stat">
            <div class="n">320</div>
            <div class="l">Knowledge Base</div>
            <div class="s">entries indexed</div>
        </div>
    </div>
    """)

    # ── QUICK QUESTIONS ──────────────────────────────────────
    gr.HTML('<div id="qq-wrap"><div class="qq-title">Quick Questions</div>')

    with gr.Row(elem_id="qq-row1"):
        btn_s1 = gr.Button("🎓 Admission process & CNC",      elem_id="btn-s1")
        btn_s2 = gr.Button("🎉 Student clubs & activities",    elem_id="btn-s2")
    with gr.Row(elem_id="qq-row2"):
        btn_s3 = gr.Button("⚙️ Engineering specializations",  elem_id="btn-s3")
        btn_s4 = gr.Button("📍 Campus location & contacts",   elem_id="btn-s4")
    with gr.Row(elem_id="qq-row3"):
        btn_s5 = gr.Button("💼 Internships & PFE",            elem_id="btn-s5")
        btn_s6 = gr.Button("💰 Tuition fees & scholarships",  elem_id="btn-s6")

    btn_reset = gr.Button("↺ Clear conversation", elem_id="btn-reset")

    gr.HTML('</div>')  # ferme qq-wrap

    # ── CHAT PANEL ───────────────────────────────────────────
    gr.HTML('<div id="chat-wrap">')

    chatbot = gr.Chatbot(
        label="", height=420,
        show_label=False, elem_id="chatwindow"
    )

    with gr.Row():
        txt_input = gr.Textbox(
            placeholder="Ask anything about ENSA Fès...",
            show_label=False, scale=5, container=False
        )
        btn_send = gr.Button("Send →", scale=1, elem_id="send-btn")

    conf_html = gr.HTML(
        value="<div style='color:#8da9c4;font-size:13px;padding:6px 0;'>"
              "Confidence score will appear here...</div>"
    )

    with gr.Row():
        with gr.Accordion("📚 Retrieved sources (top-3)", open=False):
            sources_html_out = gr.HTML(
                value="<div style='color:#8da9c4;font-size:13px;padding:8px;'>"
                      "Sources will appear here...</div>"
            )
        with gr.Accordion("📄 Raw passage", open=False):
            passage_out = gr.Textbox(
                label="", lines=3, interactive=False,
                placeholder="Raw retrieved passage..."
            )

    gr.HTML('</div>')  # ferme chat-wrap

    # ── HOW IT WORKS ─────────────────────────────────────────
    gr.HTML("""
    <div id="hiw-wrap">
        <div class="hiw-title">How It Works</div>
        <div class="hiw-grid">
            <div class="hiw-item">
                <div class="hiw-num">1</div>
                <div class="hiw-text">
                    <strong>Vector Encoding</strong>
                    <span>Your question is encoded into a 384-dimensional semantic vector</span>
                </div>
            </div>
            <div class="hiw-item">
                <div class="hiw-num">2</div>
                <div class="hiw-text">
                    <strong>FAISS Retrieval</strong>
                    <span>Top-3 closest passages retrieved from the knowledge base</span>
                </div>
            </div>
            <div class="hiw-item">
                <div class="hiw-num">3</div>
                <div class="hiw-text">
                    <strong>RoBERTa Reading</strong>
                    <span>The reader model extracts the exact answer span from passages</span>
                </div>
            </div>
            <div class="hiw-item">
                <div class="hiw-num">4</div>
                <div class="hiw-text">
                    <strong>Confidence Score</strong>
                    <span>Final score combines FAISS similarity and QA model signals</span>
                </div>
            </div>
        </div>
    </div>
    """)

    # ── FOOTER ───────────────────────────────────────────────
    gr.HTML("""
    <div id="ensa-footer">
        ENSA Fès Intelligent Chatbot &nbsp;·&nbsp;
        RAG Pipeline · multi-qa-MiniLM + RoBERTa &nbsp;·&nbsp;
        École Nationale des Sciences Appliquées · 2024–2025
    </div>
    """)

    # ── EVENTS ───────────────────────────────────────────────
    state = gr.State([])

    btn_send.click(
        repondre_v2,
        inputs=[txt_input, state],
        outputs=[chatbot, conf_html, passage_out, sources_html_out, txt_input]
    )
    txt_input.submit(
        repondre_v2,
        inputs=[txt_input, state],
        outputs=[chatbot, conf_html, passage_out, sources_html_out, txt_input]
    )

    # Boutons quick questions → remplissent la textbox
    btn_s1.click(lambda: "How to get admitted to ENSA Fes?",        outputs=txt_input)
    btn_s2.click(lambda: "What clubs are available at ENSA Fes?",    outputs=txt_input)
    btn_s3.click(lambda: "What engineering programs are available?", outputs=txt_input)
    btn_s4.click(lambda: "Where is ENSA Fes located?",               outputs=txt_input)
    btn_s5.click(lambda: "Are internships mandatory at ENSA Fes?",   outputs=txt_input)
    btn_s6.click(lambda: "What are the tuition fees at ENSA Fes?",   outputs=txt_input)

    btn_reset.click(
        lambda: (
            [], [],
            "<div style='color:#8da9c4;font-size:13px;padding:6px 0;'>"
            "Confidence score will appear here...</div>",
            "<div style='color:#8da9c4;font-size:13px;padding:8px;'>"
            "Sources will appear here...</div>"
        ),
        outputs=[chatbot, state, conf_html, sources_html_out]
    )

demo.launch(share=False, server_port=7857, show_error=True, allowed_paths=[_dir])

C:\Users\BrandlyFES\AppData\Local\Temp\ipykernel_9324\2952602053.py:221: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css) as demo:


OSError: Cannot find empty port in range: 7856-7856. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.